### Mapping human lymph node cell types to 10X Visium with Cell2location - study - implementation

This tutorial shows how to use **cell2location** method for spatially resolving fine-grained cell types by integrating 10X Visium data with scRNA-seq reference of cell types. Cell2location is a principled Bayesian model that estimates which combination of cell types in which cell abundance could have given the mRNA counts in the spatial data, while modelling technical effects (platform/technology effect, contaminating RNA, unexplained variance).

<div class="alert alert-info">
<b>Note!</b>

Cell2location is an independent package, but is powered by scvi-tools. If you have questions about cell2location, Visium data or scvi-tools please visit https://discourse.scverse.org/c/ecosytem/cell2location/42, https://discourse.scverse.org/c/general/visium/32 or https://discourse.scverse.org/c/help/scvi-tools/7 correspondingly.

</div>

### Cell2location

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BayraktarLab/cell2location/blob/master/docs/notebooks/cell2location_tutorial.ipynb)

In this tutorial, we analyse a publicly available Visium dataset of the human lymph node from 10X Genomics, and spatially map a comprehensive atlas of 34 reference cell types derived by integration of scRNA-seq datasets from human secondary lymphoid organs.

- Cell2location provides high sensitivity and resolution by borrowing statistical strength across locations. This is achieved by modelling similarity of location patterns between cell types using a hierarchical factorisation of cell abundance into tissue zones as a prior (see paper methods).

- Using our statistical method based on Negative Binomial regression to robustly combine scRNA-seq reference data across technologies and batches results in improved spatial mapping accuracy. Given cell type annotation for each cell, the corresponding reference cell type signatures $g_{f,g}$, which represent the average mRNA count of each gene $g$ in each cell type $f$, can be estimated from sc/snRNA-seq data using either 1) NB regression or 2) a hard-coded computation of per-cluster average mRNA counts for individual genes.  We generally recommend using NB regression. This notebook shows use a dataset composed on multiple batches and technologies.When the batch effects are small, a faster hard-coded method of computing per cluster averages provides similarly high accuracy. We also recommend the hard-coded method for non-UMI technologies such as Smart-Seq 2.

- Cell2location needs untransformed unnormalised spatial mRNA counts as input.

- You also need to provide cell2location with the expected average cell abundance per location which is used as a prior to guide estimation of absolute cell abundance. This value depends on the tissue and can be estimated by counting nuclei for a few locations in the paired histology image but can be approximate (see [paper methods for more guidance](https://github.com/BayraktarLab/cell2location/blob/master/docs/images/Note_on_selecting_hyperparameters.pdf)).


### Workflow diagram

see: https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_tutorial.html

## Loading packages <a class="anchor" id="Loading-packages"></a>

In [1]:
import os, sys
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

import scipy.sparse as sp
import anndata as ad
import scvi
import h5py

import cell2location
import squidpy as sq

'''
sed -i '1i import anndata' ~/uv/cell_cell_communication/.venv/lib/python3.12/site-packages/cell2location/utils/filtering.py
head -5 ~/uv/cell_cell_communication/.venv/lib/python3.12/site-packages/cell2location/utils/filtering.py

import traceback
try:
    import cell2location
except Exception:
    traceback.print_exc()
'''

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib import rcParams
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe

from scipy.stats import spearmanr
import scipy.sparse as sps

colors = ("#57a4e2", "#ebd246", "#f30707")
cmap_byr = LinearSegmentedColormap.from_list("blue_yellow_red", list(colors))

rcParams['pdf.fonttype']= 42 # enables correct plotting of text for PDFs

import warnings

ROOT_SRC = Path("/home/flavio/uv/cell_cell_communication/src")
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.Basic import pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import *
from libs.cell2location_lib import *          # clip_for_plot, create_gene_age_tissue_groups, plot_panel_umap_by_tissue_age, ...

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

squidpy==1.8.3


In [2]:
import importlib

importlib.metadata.version('scanpy'),  importlib.metadata.version('squidpy')

('1.12.4', '1.8.3')

First, let's define where we save the results of our analysis:

In [4]:
root0 = Path('/home/flavio/uv/cell_cell_communication')
root_results = root0 / 'results'
root_figure = root0 / 'figures'
root_data = root0 / 'data'

os.listdir(root_data)[:3]

['visium_hne_liana_bivariate.h5ad',
 'sc_mouse_cortex.h5ad',
 'lymphoid_organ_sc.h5ad']

### Loading Edited Visium and scRNA-seq reference data <a class="anchor" id="Loading-Visium-and-scRNA-seq-reference-data"></a>

First let's read spatial Visium data from 10X Space Ranger output. Here we use lymph node data generated by 10X and presented in [Kleshchevnikov et al (section 4, Fig 4)](https://www.biorxiv.org/content/10.1101/2020.11.15.378125v1). This dataset can be conveniently downloaded and imported using scanpy. See [this tutorial](https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_short_demo.html) for a more extensive and practical example of data loading (multiple visium samples).

### Visium data - adata_vis

In [ ]:
results_folder =  root_results / 'lymph_nodes_analysis/'

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = Path(f'{results_folder}/reference_signatures')
run_name = Path(f'{results_folder}/cell2location_map')

adata_file_sp = run_name / "sp.h5ad"
adata_file_sp.exists(), adata_file_sp

In [ ]:
adata_vis = sc.read_h5ad(adata_file_sp)
print(f"adata_vis read from {adata_file_sp}")

# Reload the trained spatial model bound to that AnnData
mod_c2l = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)
print(f"Model loaded from {run_name}")

# mod_c2l.samples is not saved with the model: rebuild it (needed by plot_QC and compute_expected_per_cell_type)
adata_vis = ensure_posterior_samples(mod_c2l, adata_vis, quantiles=["q05", "q50", "q95", "q0001"], accelerator="gpu")
print("posterior samples:", sorted(mod_c2l.samples.keys()))

In [ ]:
adata_vis

In [ ]:
print(adata_vis.var.shape[0])
adata_vis.var.head(3)

In [ ]:
adata_vis.uns['spatial'].keys()

In [ ]:
adata_vis.obs.head(5).T.head(8)

In [ ]:
adata_vis.var_names[:5]

In [ ]:
print(adata_vis.var.shape)
adata_vis.var.head(5)

In [ ]:
"DPM1" in list(adata_vis.var.symbol)

In [ ]:
genes = [x for x in adata_vis.var.symbol if x.startswith("CD")]
genes.sort()
np.array(genes)[:20]

### B-cell gene

Pan B-cell gene and protein markers used to identify B lymphocytes include CD19, MS4A1 (which encodes CD20), CD79A, CD79B, PAX5, and CD22. [1](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [2](https://www.panglaodb.se/markers.html?cell_type='B%20cells'), [3](https://www.rpubs.com/daniloceschin/Markers_BCells)

Key B-Cell Gene and Surface Markers

* Core Pan B-Cell Markers:
  * CD19: Expressed from the early pro-adata_bcell stage through mature B cells; acts as a coreceptor for B-cell receptor (BCR) signaling.
  * MS4A1 (CD20): Encodes the CD20 calcium channel protein; present from the pre-B stage to mature B cells and a primary target for immunotherapies (e.g., rituximab).
  * CD79A / CD79B: Encodes Igα and Igβ, essential signaling components associated with the surface B-cell receptor complex.
  * PAX5: A master transcription factor critical for B-cell commitment and identity.
  * CD22: A sialic acid-binding lectin expressed on mature and developing B cells. [1](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/), [2](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [3](https://www.abcam.com/en-us/technical-resources/research-areas/marker-guides/b-cell-markers), [4](https://www.youtube.com/watch?v=JUq3K8rtcso&vl=en&t=178), [5](https://www.origene.com/research-areas/b-cell-lineage), [6](https://www.rpubs.com/daniloceschin/Markers_BCells), [7](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/)
* Stage-Specific and Subset Markers:
  * Naïve B Cells: IGHD (IgD), IGHM (IgM), CD27−, IL4R, FCER2 (CD23).
  * Memory B Cells: CD27+, CD24+, and switched or unswitched immunoglobulins (IgG, IgA, IgE).
  * Plasma Cells / Plasmablasts: Downregulate CD19 and MS4A1/CD20 while upregulating CD38, SDC1 (CD138), PRDM1 (Blimp-1), and XBP1. [1](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [2](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [3](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/)

### See also

notebook: ccc00b_immune_cells.ipynb

### lymphoid_organ_sc.h5ad - reference

**lymphoid_organ_sc.h5ad** is the single-cell reference for the cell2location paper's **lymph node / tonsil analysis**. 

It holds **73,260 cells × 10,237 genes**, 1.66 GB on disk, and contains both raw counts and a fitted reference model. I opened it read-only in backed mode, so the file was not changed.

#### Expression matrix (**X**)

- **Raw UMI counts.** The matrix is sparse (CSR, float32) and every value is a whole number. In the first 2,000 cells the counts run from 1 to 1,276, with about 1,080 detected genes per cell.

- **Genes are already filtered.** 

  - Gene symbols are the index and Ensembl IDs are in **var['GeneID-2']**. 

  - The **n_counts** and **n_genes** columns were computed on the full gene set before filtering, so they are slightly higher than the row sums of **X** (e.g. 3,148 vs 3,122).

- **Cell QC:** 
  - median 3,483 UMIs and 1,657 genes per cell; 

  - minimum 581 UMIs and 698 genes.
  - Mitochondrial fraction (**percent_mito**) is 0–15%, median 4%. 
  - Every cell has **predicted_doublet = False**, so doublets were already removed.

#### Studies and tissues (**obs**)

| Study | Tissue | 3′ (3GEX) | 5′ (5GEX) |
|---|---|---|---|
| King_2020 | Tonsil | 2,002 | 27,703 |
| James_2020 | Mesenteric LN | 10,250 | 4,184 |
| Park_2020 | Spleen / iLN / mLN / tLN | 15,506 | 13,615 |

- **23 samples from 13 donors.** Donor A16 alone contributes 29,121 cells.

- **Ages** run from 0–4 years to 65–70 years.
- **Sex:** 48,545 female and 24,715 male cells.
- **Extra columns:** B-cell receptor fields (**BCELL_CLONE**, **ISOTYPE**, **IGH_MU_FREQ**, **VDJsum**), cell-cycle scores and phase, five Leiden clusterings, and several generations of annotation columns.


1. King HW, Orban N, Riches JC, Clear AJ, Warnes G, Teichmann SA, James LK. Single-cell analysis of human B cell maturation predicts how antibody class switching shapes selection dynamics. Sci Immunol. 2021 Feb 12;6(56):eabe6291. doi: 10.1126/sciimmunol.abe6291. PMID: 33579751.

2. James KR, Gomes T, Elmentaite R, Kumar N, Gulliver EL, King HW, Stares MD, Bareham BR, Ferdinand JR, Petrova VN, Polański K, Forster SC, Jarvis LB, Suchanek O, Howlett S, James LK, Jones JL, Meyer KB, Clatworthy MR, Saeb-Parsy K, Lawley TD, Teichmann SA. Distinct microbial and immune niches of the human colon. Nat Immunol. 2020 Mar;21(3):343-353. doi: 10.1038/s41590-020-0602-z. Epub 2020 Feb 17. PMID: 32066951; PMCID: PMC7212050.

Gastrointestinal microbiota and immune cells interact closely and display regional specificity, but little is known about how these communities differ with location. Here, we simultaneously assess microbiota and single immune cells across the healthy, adult human colon, with paired characterization of immune cells in the mesenteric lymph nodes, to delineate colonic immune niches at steady-state. We describe distinct T helper cell activation and migration profiles along the colon and characterize the transcriptional adaptation trajectory of T regulatory cells between lymphoid tissue and colon. Finally, we show increasing B cell accumulation, clonal expansion and mutational frequency from cecum to sigmoid colon, and link this to the increasing number of reactive bacterial species.

3. Park JE, Botting RA, Domínguez Conde C, Popescu DM, Lavaert M, Kunz DJ, Goh I, Stephenson E, Ragazzini R, Tuck E, Wilbrey-Clark A, Roberts K, Kedlian VR, Ferdinand JR, He X, Webb S, Maunder D, Vandamme N, Mahbubani KT, Polanski K, Mamanova L, Bolt L, Crossland D, de Rita F, Fuller A, Filby A, Reynolds G, Dixon D, Saeb-Parsy K, Lisgo S, Henderson D, Vento-Tormo R, Bayraktar OA, Barker RA, Meyer KB, Saeys Y, Bonfanti P, Behjati S, Clatworthy MR, Taghon T, Haniffa M, Teichmann SA. A cell atlas of human thymic development defines T cell repertoire formation. Science. 2020 Feb 21;367(6480):eaay3224. doi: 10.1126/science.aay3224. PMID: 32079746; PMCID: PMC7611066.

The thymus provides a nurturing environment for the differentiation and selection of T cells, a process orchestrated by their interaction with multiple thymic cell types. We used single-cell RNA sequencing to create a cell census of the human thymus across the life span and to reconstruct T cell differentiation trajectories and T cell receptor (TCR) recombination kinetics. Using this approach, we identified and located in situ CD8αα+ T cell populations, thymic fibroblast subtypes, and activated dendritic cell states. In addition, we reveal a bias in TCR recombination and selection, which is attributed to genomic position and the kinetics of lineage commitment. Taken together, our data provide a comprehensive atlas of the human thymus across the life span with new insights into human T cell development.


#### Cell-type annotations

- ****Subset** (34 types)** is the label set the regression model was trained on. The largest groups are 
  - B_mem (13,476), 
  - B_naive (8,924),
  - T_CD4+_naive (6,012),
  - B_Cycling (4,765) and
  - T_CD4+_TfH (4,690). 
  - It also covers
    - germinal-centre B cells (DZ, LZ, prePB, preGC), 
    - T-cell types (TfH, Tfr, Treg, CD8 types), 
    - NK/NKT/ILC, 
    - myeloid cells (DCs, macrophages, monocytes, mast cells) and 
    - stromal cells (Endo, FDC, VSMC). 
    - Rare types include ILC/Mast (18), VSMC (40) and DC_CCR7+ (42).

- ****Subset_Broad** (38 types)** is finer, despite its name. 
  - For example, it splits the endothelium into ACKR1+, CXCL12+ and lymphatic cells.

- **Likely label swap:** 
  - **Subset** has ILC = 617 and Mast = 18, 
  - while **Subset_Broad** has Mast = 617 and ILC = 18. 
  - I only compared the counts and did not cross-tabulate cell by cell. Check marker genes (e.g. TPSAB1/KIT for mast cells) before relying on either label for these two types.

#### Fitted reference model

This file already contains the trained cell2location reference model, so you don't need to retrain it.

- **uns['regression_mod']** is a **RegressionNBV4Torch** model with 57 factors: 
  - 34 cell types plus 
  - 23 sample effects.
- Its posterior means are:
  - **gene_factors** (57 × 10,237)
  - **sample_scaling** (23 × 1)
  - **tech_scaling** (2 × 10,237, one row per chemistry, 3′ and 5′)
  - **gene_E** (1 × 10,237)
- **In **var**:** 34 **mean_cov_effect_Subset_*** columns hold each cell type's expression signature. These are the columns to pass to cell2location as the reference. There are also 23 **mean_sample_effectSample_*** columns, plus **n_cells** and **nonz_mean**.

#### Embeddings and metadata

- **Embeddings:** **obsm['X_pca']** (100 components), **obsm['X_umap']**, and a neighbour graph built with 69 neighbours (**obsp** connectivities and distances).
- **Colour palettes** are stored in **uns** for the main categories.
- **Unusable columns:** **GeneName-2** holds numbers instead of gene names, and the **feature_types** / **gene_ids-*** columns are mostly NaN. Use the var index for gene names instead.

To extract the signatures for spatial mapping:
```python
sig = adata.var[[c for c in adata.var if c.startswith('mean_cov_effect_Subset_')]]
sig.columns = sig.columns.str.replace('mean_cov_effect_Subset_', '')
```

This was a description only, so I didn't save any files.

In [ ]:
fname_sc = 'lymphoid_organ_sc.h5ad'
filename = root_data / fname_sc

print(f"{os.path.getsize(filename)/1e9:.3f} GB")
filename.exists(), filename

In [ ]:
adata_ref = ad.read_h5ad(filename)
print(f"Data read from {filename}")

adata_ref

In [ ]:
X=adata_ref.X
print(type(X), getattr(X,'dtype',None))

with h5py.File(filename,'r') as f:
    g=f['X']
    print(dict(g.attrs) if isinstance(g,h5py.Group) else g.shape)
    for k in f['layers'].keys() if 'layers' in f else []: print('layer',k)

### Which genes are unique when comparing the B cell types? post-study

GC DZ, GC LZ, GC prePB, B cycling, preGC, naive, activated, B IFN, B mem, and B plasma

In [ ]:
warnings.filterwarnings("ignore")

btypes=["B_GC_DZ","B_GC_LZ","B_GC_prePB","B_Cycling","B_preGC","B_naive","B_activated","B_IFN","B_mem","B_plasma"]

adata_bcell = adata_ref[adata_ref.obs.Subset.isin(btypes)].copy()

adata_bcell.obs["Subset"]=pd.Categorical(adata_bcell.obs.Subset.astype(str),categories=btypes)
print(adata_bcell.shape, adata_bcell.obs.Subset.value_counts().reindex(btypes).to_dict())

adata_bcell.layers["counts"]=adata_bcell.X.copy()

sc.pp.normalize_total(adata_bcell,target_sum=1e4)
sc.pp.log1p(adata_bcell)

# drop genes expressed in <1% of adata_bcell cells to speed up
keep=np.asarray((adata_bcell.layers["counts"]>0).mean(0)).ravel()>=0.01
adata_bcell=adata_bcell[:,keep].copy()
print("genes tested:",adata_bcell.n_vars)

sc.tl.rank_genes_groups(adata_bcell,"Subset",method="wilcoxon",pts=True,use_raw=False)

# per-group df_pct and mean (log1p) for specificity vs every other adata_bcell subset
Xb=(adata_bcell.layers["counts"]>0)
grp=adata_bcell.obs.Subset.values

df_pct=pd.DataFrame({g:np.asarray(Xb[(grp==g)].mean(0)).ravel() for g in btypes},index=adata_bcell.var_names)
mexp=pd.DataFrame({g:np.asarray(np.expm1(adata_bcell.X[(grp==g)]).mean(0)).ravel() for g in btypes},index=adata_bcell.var_names)  # mean CP10k
df_pct.to_pickle(root_results / "b_pct.pkl"); mexp.to_pickle(root_results / "b_mean.pkl")

pickle.dump(adata_bcell.uns["rank_genes_groups"],open(root_results / "b_rgg.pkl","wb"))
adata_bcell.obs[["Subset","Donor","Study","Tissue"]].to_pickle(root_results / "b_obs.pkl")

print("done")

In [ ]:
btypes

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
rgg=adata_bcell.uns["rank_genes_groups"]

# CP10k pseudocount
eps=0.05

rows=[]
for btype in btypes:
    other_bcells=[o for o in btypes if o!=btype]
    dfw = c2l.wtab(btype, rgg)

    dic = {"pct_in": df_pct[btype], "max_pct_other": df_pct[other_bcells].max(1),
           "next_pct_type": df_pct[other_bcells].idxmax(1),
           "mean_cp10k": mexp[btype],"max_mean_other": mexp[other_bcells].max(1),
           "next_mean_type": mexp[other_bcells].idxmax(1)}
    
    df=pd.DataFrame(dic)
    df["log2fc_vs_next"]=np.log2((df.mean_cp10k+eps)/(df.max_mean_other+eps))
    df=df.join(dfw)
    df["group"]=btype
    rows.append(df.reset_index().rename(columns={"index":"gene"}))

dft=pd.concat(rows)

strict=(dft.padj<0.05)&(dft.pct_in>=0.30)&(dft.max_pct_other<=0.25)&(dft.log2fc_vs_next>=1)
relax =(dft.padj<0.05)&(dft.pct_in>=0.30)&(dft.max_pct_other<=0.50)&(dft.log2fc_vs_next>=0.58)&~strict

dft["tier"]=np.where(strict,"unique",np.where(relax,"enriched (shared)",""))

# donor consistency for candidates
obs=adata_bcell.obs
donors=obs.Donor.astype(str).values
grp=obs.Subset.astype(str).values

cand=dft[dft.tier!=""]
Xn=adata_bcell.X  # log1p

cnt=pd.crosstab(donors,grp)

dc=[c2l.donor_consistency(adata_bcell, r.gene,r.group, btypes, donors, grp, Xn, cnt) for r in cand.itertuples()]
dft.loc[dft.tier!="","donors_top"]=[f"{a}/{b}" for a,b in dc]

dff=dft[dft.tier!=""].sort_values(["group","tier","log2fc_vs_next"],ascending=[True,False,False])

dff["group"]=pd.Categorical(dff.group,btypes)
dff=dff.sort_values(["group","tier","log2fc_vs_next"],ascending=[True,False,False])

cols=["group","gene","tier","pct_in","max_pct_other","next_pct_type",
      "log2fc_vs_next","next_mean_type","lfc_vs_restB","padj","donors_top"]

dff[cols].round(3).to_csv(root_results / "bcell_subset_unique_markers.csv",index=False)

dff_summ=dff.groupby(["group","tier"],observed=True).size().unstack(fill_value=0)
print(dff_summ)

for btype in btypes:
    dff_u = dff[(dff.group==btype)&(dff.tier=="unique")].head(12)
    dff_e = dff[(dff.group==btype)&(dff.tier!="unique")].head(6)

    print(f"\n## {btype}\n U:", ", ".join(f"{r.gene}({r.pct_in*100:.0f}/{r.max_pct_other*100:.0f}%,{r.log2fc_vs_next:.1f},{r.donors_top})" for r in dff_u.itertuples()))
    print(" E:", ", ".join(f"{r.gene}({r.pct_in*100:.0f}/{r.max_pct_other*100:.0f}%,{r.log2fc_vs_next:.1f} vs {r.next_mean_type})" for r in dff_e.itertuples()))

cnt.reindex(columns=btypes)


In [ ]:
dff.head(3)

In [ ]:
# summary table for the response
cols=["group","gene","pct_in","max_pct_other","log2fc_vs_next","donors_top"]

dftab=dff[dff.tier=="unique"].groupby("group",observed=True)[cols]
dftab.head(10)

In [ ]:
for btype in ["B_naive", "B_mem", "B_GC_LZ", "B_activated"]:

    dfa=dft[(dft.group==btype) & (dft.padj<0.05) & (dft.pct_in>=0.3)]
    dfa = dfa.sort_values("log2fc_vs_next",ascending=False).head(8)
    print(btype, "| best vs next:", ", ".join(f"{r.gene}({r.pct_in*100:.0f}/{r.max_pct_other*100:.0f}%,{r.log2fc_vs_next:.2f} vs {r.next_mean_type})" for r in dfa.itertuples()))
    
    dfw = c2l.wtab(btype, rgg).head(8)
    print("   top Wilcoxon vs rest-B:", ", ".join(f"{i}({row.lfc_vs_restB:.1f})" for i,row in dfw.iterrows()))

# naive vs mem head to head
sc.tl.rank_genes_groups(adata_bcell,"Subset",groups=["B_naive"],reference="B_mem",method="wilcoxon",key_added="nm",use_raw=False)
nm=sc.get.rank_genes_groups_df(adata_bcell,"B_naive",key="nm")

print("")
print("naive>mem:", nm.head(10)[["names","logfoldchanges"]].round(2).values.tolist())
print("mem>naive:", nm.tail(10)[["names","logfoldchanges"]].round(2).values.tolist()[::-1])

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

markers={"B_GC_DZ":["SUGCT","MME","SH3TC1","USP53"],
 "B_GC_LZ":["LMO2","RGS13","GMDS","PLEK"],
 "B_GC_prePB":["FRZB","BAIAP2L1","ARHGAP10","RASSF6"],
 "B_Cycling":["CDK1","TOP2A","RRM2","UBE2C"],
 "B_preGC":["CCND2","MIR155HG","CRIP3","LILRA4"],
 "B_naive":["TCL1A","IGHD","FCER2","IGHM"],
 "B_activated":["CD69","CD83","NR4A1","NFKBIZ"],
 "B_IFN":["IFIT3","IFIT1","IFI44L","MX2"],
 "B_mem":["CD27","TNFRSF13B","GPR183","CLECL1"],
 "B_plasma":["DERL3","FKBP11","IGHG2","PRDX4"]}


dp = c2l.dotplot_bcell_markers(adata_bcell, markers, root_figure)


### Comparing the 10 B-cell subsets against each other only

Not against T or myeloid cells. That way a gene counts as "unique" only if it separates one B subset from all the other B subsets. Per subset, I'll combine:

1. a one-vs-rest Wilcoxon test within the B-cell compartment (log1p CP10k);
2. a stricter specificity filter: the gene must beat the *next-highest* B subset by ≥ 1 log2 fold, be detected in ≥ 30% of the subset's cells, and in ≤ 25% of cells of every other B subset;
3. a donor-consistency check, because cell-level tests overstate significance when cells come from few donors.

The test ran on 38,309 B cells and 8,375 genes. Subset sizes range from 74 (`B_GC_prePB`) to 13,476 (`B_mem`). Now applying the specificity filter and the donor check:

Three subsets (B_naive, B_mem, B_GC_LZ) lack a gene passing the strict filter, so I'll find what best distinguishes them from their nearest neighbors and then build a dot plot of the selected markers.

Naive markers like TCL1A, IGHD, and FCER2 only stand out when naive cells are compared directly against memory cells, since activated, pre-GC, and IFN B subsets also carry them in the full 10-way comparison. I'll generate the dot plot next.

Six of the 10 B-cell subsets have clearly unique genes: plasma, cycling, IFN, pre-plasmablast (prePB), pre-GC and dark zone (DZ). Naive B, memory B and GC light zone (LZ) have **no** gene that is unique against all the other B subsets. Activated B has only one marginal one (*MYADM*). These four are defined by combinations of genes, or by contrast with one neighbouring subset.

![Markers separating the 10 B-cell subsets](a86a6a63-0874-43cf-9da5-8583b62b3b10)

I compared the B subsets only against each other: 38,309 B cells, 8,375 genes, log1p(CP10k), one-vs-rest Wilcoxon test. A gene counts as **unique** if it meets all of these:
- adjusted p < 0.05;
- detected in ≥ 30% of the subset's cells;
- detected in ≤ 25% of cells in *every* other B subset;
- at least 2× the mean of the next-highest B subset.

"Donors" is the number of donors (each with ≥ 20 cells of that subset) in which the gene is still highest in that subset.

| Subset (cells) | Unique genes (top) | % in subset / max % in other B subsets | Donors |
|---|---|---|---|
| **B_plasma** (1,094) | *DERL3, FKBP11, IGHG2, IGHGP, PRDX4, TXNDC5, FNDC3B* | *DERL3* 92 / 18 | 6/6 |
| **B_Cycling** (4,765) | *CDK1, TOP2A, RRM2, UBE2C, CCNA2, TYMS, ASF1B* (132 genes in total) | *CDK1* 56 / 4 | 8/8 |
| **B_IFN** (199) | *IFIT3, IFIT1, IFI44L, IFI6, MX2, STAT1, XAF1* | *IFIT3* 56 / 2 | 2/2 |
| **B_GC_prePB** (74) | *FRZB, BAIAP2L1, ARHGAP10, BTNL9, CNR1* | *BAIAP2L1* 39 / 7 | 1/1 |
| **B_preGC** (404) | *CCND2, MIR155HG, CRIP3, SLAMF1, LILRA4* | *CCND2* 37 / 13 | 5/5 |
| **B_GC_DZ** (2,500) | *SH3TC1, USP53, SLC2A5* (unique); *SUGCT, MME* (CD10), shared with cycling | *SH3TC1* 36 / 16 | 8/8 |
| **B_activated** (3,575) | *MYADM* only; otherwise immediate-early genes *CD69, CD83, FOS, NR4A1, NFKBIZ* | *MYADM* 37 / 21 | 7/9 |
| **B_GC_LZ** (3,298) | none; best are *LMO2* (51 / 27%, 2.8× DZ), *RGS13, GMDS, PLEK* (shared with DZ, prePB, cycling) | — | — |
| **B_mem** (13,476) | none; best are *TNFRSF13B* (TACI), *CLECL1, GPR183* (shared with IFN B) | — | — |
| **B_naive** (8,924) | none; *IGHD, FCER2, TCL1A, IGHM* are shared with the activated, pre-GC and IFN B cells | — | — |


**Naive vs memory, head to head:** *TCL1A* (log2FC 5.5), *IGHD* (3.2), *FCER2* (2.5) and *CD72* are higher in naive cells. *TNFRSF13B* (3.4), *CD27* (3.3) and *GPR183* (2.4) are higher in memory cells. So the practical rule for naive B is IGHD⁺ FCER2⁺ TCL1A⁺ CD27⁻.

**Caveats:**
- **Cycling and IFN describe a cell state, not a lineage.** Their unique genes are the cell-cycle and interferon programs. DZ cells are proliferating too, which is why *SUGCT* and *MME* are shared with `B_Cycling` and not unique to DZ.
- **Some subsets are too small to check across donors.** `B_GC_prePB` has 74 cells, and only one donor has ≥ 20 of them; `B_IFN` has 199 cells, and only two donors qualify. Their markers are consistent only within those few donors.
- ***LILRA4* in pre-GC B is suspicious.** It is a classic plasmacytoid-DC marker, so it may come from contaminating cells or doublets, not genuine B-cell expression.
- **The p-values are optimistic.** The test treats each cell as independent, although cells come from only 13 donors. Rely on the effect-size filters and the donor counts more than on the p-values.
- ***TCL1A* and *CD27* aren't specific to naive or memory cells.** In the dot plot *TCL1A* is highest in GC DZ and cycling cells, and *CD27* is highest in plasma cells. Each is specific only in the naive-vs-memory contrast.
- **The four-genes-per-subset choice was mine.** I picked the dot-plot genes from the unique list where one existed, and otherwise from the best shared or head-to-head genes.

Files are in the output tray:
- `bcell_subset_markers_dotplot.png` — the dot plot above.
- `bcell_subset_unique_markers.csv` — 502 candidate markers in two tiers: "unique", and "enriched (shared)", where the gene is ≥ 1.5× the next subset and detected in ≤ 50% of every other subset. For each gene it gives the % in the subset, the max % in other subsets, the next-highest subset, log2FC vs that subset, adjusted p and the donor count.

### new genes_plot: selected mannually

In [ ]:
# Memory B Cells: CD27+, CD24+,
# Core Pan B-Cell: CD19, MS4A1 (CD20):, CD79A / CD79B, PAX5, CD22
# Naïve B Cells: IGHD (IgD), IGHM (IgM), CD27−, IL4R, FCER2 (CD23)

genes_plot = ["AICDA", "CCR7", 
              "CD19", "MS4A1", "CD21", "CD22", "CD23", "CD24", "CD27", "CD28",
              "CD38", "CD52", "CD53", "CD79A", "CD79B", "CDS2",
              "FCER2", "IGHD", "IGHM", "IL4R", "LEF1", "SELL", "PAX5",  "TCF7",
              # extra to obser in cell type signature
              "DERL3", "CDK1", "IFIT3", "IFIT1", "BAIAP2L1", "ARHGAP10", "FRZB", "CCND2", "SH3TC1", "USP53", "SLC2A5", "MYADM",
              "TNFRSF13B", "CLECL1", "GPR183", "IGHD", "FCER2", "TCL1A", "IGHM", "RRM2", "HIST1H3B", "LILRA4", "CRIP3",
              "MYADM", "IGHG2", "IGHGP", "DERL3"]

genes_plot = np.unique(genes_plot)

print("Genes to plot", len(genes_plot))

genes = [x for x in genes_plot if x in list(adata_vis.var.symbol)]
genes = np.unique(genes)

len(genes), "; ".join(genes)

### Visium spatial plot per gene

- see filtered markers in: result/bcell_subset_unique_markers.csv 

In [ ]:
# clip_for_plot / _values live in libs/cell2location_lib.py
# (warns about unknown features; non-zero percentile cap for very sparse genes such as CD14)
genes, cols = clip_for_plot(adata_vis, genes, q=99.2, vmin=0)
print(len(genes))

In [ ]:
from IPython.core.display import HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# pd.reset_option('display.max_colwidth')
# pd.reset_option('display.max_columns')

pd.set_option('display.max_colwidth', 200)

In [ ]:
fname = 'selected_genes_lymphoid_organ.csv'
df_genes = pdreadcsv(fname, root_results, sep=',')
df_genes

In [ ]:
%matplotlib inline

nframe=8

for i in range(0, len(genes), nframe):
    # print(i, i+nframe)

    sq.pl.spatial_scatter(
        adata_vis,
        color=cols[i:(i+nframe)],
        title=list(genes[i:(i+nframe)]),            # show original names, not the _plot_ prefix
        img_res_key="hires",
        img_alpha=0.6,
        size=1.3,
        cmap=cmap_byr,
        ncols=4,
        library_id="V1_Human_Lymph_Node",
    )

### 

### find mitochondria-encoded (MT) genes

In [ ]:
adata_vis[:, adata_vis.var['MT_gene'].values]

In [ ]:
adata_vis.var[adata_vis.var['symbol'].str.startswith('MT-')]

In [ ]:
adata_vis.obsm['MT']

### edited adata_ref

Published scRNA-seq datasets of lymph nodes have typically lacked an adequate representation of germinal centre-associated immune cell populations due to age of patient donors. We, therefore, include scRNA-seq datasets spanning lymph nodes, spleen and tonsils in our single-cell reference to ensure that we captured the full diversity of immune cell states likely to exist in the spatial transcriptomic dataset.

Here we download this dataset, import into anndata and change variable names to ENSEMBL gene identifiers.

In [ ]:
adata_ref

In [ ]:
print(adata_ref.var.shape)
list(adata_ref.var.columns)[:10]

In [ ]:
adata_ref.var.head(3)

In [ ]:
if 'symbol' not in adata_ref.var.columns:
    adata_ref.var['symbol']= adata_ref.var.index
    # rename 'GeneID-2' as necessary for your data
    adata_ref.var.set_index('GeneID-2', drop=True, inplace=True)

# delete unnecessary raw slot (to be removed in a future version of the tutorial)
del adata_ref.raw

adata_ref.var.head(5).T.head(6)

In [ ]:
obs=adata_ref.obs

for c in ['Study','Tissue','Method','Donor','Sample','Age','Sex','Subset_Broad','Subset','phase','predicted_doublet']:
    vc=obs[c].value_counts(dropna=False)
    print(f"\n## {c} ({vc.size} levels)")
    print(vc.head(40).to_string())


In [ ]:
cols = ['n_counts','n_genes','percent_mito','doublet_score']
obs[cols].describe().round(2)

In [ ]:
# X values sample
sub=adata_ref.X[:2000]
d=sub.data

print("X sample: min",d.min(),"max",d.max(),"integer?",np.allclose(d,np.round(d)),"nnz/cell",sub.nnz/2000)
print()
print("rowsum vs n_counts", np.asarray(sub.sum(1)).ravel()[:5], obs.n_counts.values[:5])
print()
print('First five genes:', adata_ref.var.index[:5].tolist())
print()
print(adata_ref.uns['regression_mod'].keys() if hasattr(adata_ref.uns['regression_mod'],'keys') else adata_ref.uns['regression_mod'])
print()
print({k:(adata_ref.uns[k] if k!='leiden' else adata_ref.uns[k]) for k in ['neighbors']})
print()

In [ ]:
adata_ref.var[['symbol', 'GeneName-2','n_cells','nonz_mean']].head()

### Split by Age-group and Tissues

First I'll check how tissue and age overlap in the scRNA reference, and look in the squidpy project for the **genes** list and the **adata_vis** object.

Note: tissue and age are confounded. Tonsil is the only babies/young tissue, and spleen, iLN and tLN are all adult. Let me find the notebook that defines **genes** and **adata_vis**.

This Visium slide has no tissue/age labels, so I'll instead build the B-cell gene panel on the scRNA reference, coloring UMAPs by each gene split by tissue × age group with the same 99.2-percentile clipping. First, I'll load the figure-style skill.

Since CD20 isn't a valid gene symbol (the gene is MS4A1), I'll plot MS4A1 labelled as "MS4A1 (CD20)" instead. Next I'll build the grid with genes in rows and tissue × age groups in columns, using a shared colour scale per gene.

I found two issues to fix before saving: colliding column headers, and a blank CD14 column caused by its extreme sparsity flattening the 99.2-percentile clip to 0. I'll fix the headers and adjust sparse genes like CD14 to use the 99.2th percentile of non-zero values instead.

Yes, but only on the scRNA reference, not on the Visium slide. **V1_Human_Lymph_Node** is a single lymph node section from one donor, so its spots carry no tissue or age labels. Tissue and age exist only in **lymphoid_organ_sc.h5ad**. So I drew the same gene panel and the same 99.2-percentile clipping on the reference UMAP, with one column per tissue × age group.

### B-cell gene panel on the scRNA reference UMAP, split by tissue and age group

**Age groups.** Your cut-offs map onto the reference's age bins like this:

| Group | Age bins | Cells |
|---|---|---|
| babies | 0–4y | 16,237 |
| young | 5–9y | 13,468 |
| adult | 20–25y | 29,121 |
| old adult | 40–44y | 1,825 |
| elder | 55–60y, 60–64y, 65–70y | 12,609 |

**Tissue and age are almost entirely confounded.** Only 8 of the 25 tissue × age combinations contain cells:

- Tonsil exists only as babies and young.

- Spleen, iLN and tLN are adult only, and all come from the single donor A16.
- Only mLN spans three age groups: adult, old adult (one donor, 1,825 cells) and elder (four donors).

So a difference between columns is just as likely to come from tissue, study (King, Park or James) or donor as from age.

**How the plot differs from your Visium version:**

- **CD20 is replaced by MS4A1.** **CD20** is not a gene symbol, so your **clip_for_plot** drops it without a warning, and it is probably missing from your Visium plot too. I plotted **MS4A1**, labelled "MS4A1 (CD20)".

- **CD14 needed a different cap.** More than 99.2% of cells have zero *CD14*, so the 99.2-percentile cap was 0 and the row came out blank. For that gene I clipped at the 99.2th percentile of the non-zero values instead. The same problem can hit any sparse gene in your Visium helper.
- **Expression is normalised.** The reference holds raw counts from mixed 3′ and 5′ chemistry, so I coloured by log1p(counts per 10k), computed per cell. Each gene has one colour scale shared across all columns, so columns can be compared directly.
- **The colour map is sequential.** I used a light-grey-to-red ramp instead of **RdBu_r**, because expression has no meaningful midpoint for a diverging map to centre on.

If you want the split on the slide itself, one option is to build separate cell2location reference signatures per group and map each onto the Visium spots. Given the confounding, though, the result would mostly reflect tissue and study, not age.

Files are in the output tray:

- **bcell_genes_umap_by_tissue_age.png** — the figure above.

- **bcell_genes_by_tissue_age.csv** — per gene and group: cell count, % of cells expressing, and mean log1p(CP10k).

### Tissue types

In [ ]:
np.unique(adata_ref.obs["Tissue"])

### mLN only: naive / memory / GC markers by age group (adult, old adult, elder)

- genes are looked up by symbol

In [ ]:
# mLN only: naive / memory / GC markers by age group (adult, old adult, elder)
# genes are looked up by symbol (var_names are Ensembl IDs after cell 34)
ml = adata_ref[adata_ref.obs["Tissue"] == "mLN"].copy()
ml.obs["age_group"] = ml.obs["Age"].map(age_group)          # age_group() from libs.cell2location_lib

sc.pp.normalize_total(ml, target_sum=1e4)
sc.pp.log1p(ml)

# mesentherical lymphoid nodes (LN)
mln_genes, mln_ids = resolve_gene_ids(ml, ["LEF1", "CCR7", "SELL", "TCF7", "CD27", "CD28", "AICDA"])
Xm = ml[:, mln_ids].X
Xm = Xm.toarray() if sp.issparse(Xm) else np.asarray(Xm)

print(">> found genes", mln_genes, '\n')

df_mln = pd.DataFrame(Xm, columns=mln_genes, index=ml.obs_names)
df_mln["age_group"] = ml.obs["age_group"].values
mln_order = ["adult", "old adult", "elder"]

del ml, Xm

df_mln.groupby("age_group")[mln_genes].mean().reindex(mln_order).round(3)

In [ ]:
(df_mln[mln_genes] > 0).groupby(df_mln["age_group"]).mean().reindex(mln_order).round(3)

In [ ]:
print(">>> df_mln is a subset from X")
print(df_mln.shape)
df_mln.head(3)

### My selected genes - to observe them inside the lymphonode

In [ ]:
found, ids = resolve_gene_ids(adata_ref, genes_plot)   # works with symbol or Ensembl var_names
print(len(found), 'of', len(genes_plot), 'genes found')
";".join(found)

### Below one calculate the UMAP's regions

- GC DZ: dark zone (DZ) of a germinal center (GC)

- GC LZ: germinal center (GC) light zone (LZ)
- preGC: Pre-germinal center (preGC) B cells
  - refer to B-cell subsets and precursor states that exist before they enter the germinal center (GC) reaction to undergo somatic hypermutation and class-switch recombination
- FDCs: Follicular dendritic cells
- B-cell cycling 
  - refers to the process by which B lymphocytes transition from a resting state into active proliferation, DNA replication, and division during immune responses or development
- CD4+ Tfh (T follicular helper) cells are a specialized subset of CD4+ helper T cells 
  - that reside primarily in lymphoid tissues and orchestrate germinal center (GC) reactions.
- TIM3+ refers to cells that express TIM-3 (T-cell immunoglobulin and mucin-domain containing-3)
  - an immune checkpoint protein found on various immune and tumor cells
  - Receptor Type: An inhibitory immune checkpoint receptor that acts as a negative regulator of the immune system.
  - Cell Expression: Found on exhausted CD8+ T cells, CD4+ T cells, regulatory T cells (Tregs), natural killer (NK) cells, M2 macrophages, and certain tumor cells.
  - Ligands: Interacts with molecules like galectin-9 (GAL-9), CEACAM1, HMGB1, and phosphatidylserine (PtdSer).

In [ ]:
filefig = root_figure / "celltype_umap_dotplot.png"

if filefig.exists():
    plt.figure(figsize=(20,12))
    img = plt.imread(filefig)
    plt.imshow(img)


In [ ]:
apply_figure_style(sizes=(8, 7, 6))

fig = plot_panel_umap_by_tissue_age(
    adata_ref, genes_plot,
    out_png = root_figure / 'bcell_genes_umap_by_tissue_age.png',
    labels={'MS4A1': 'MS4A1 (CD20)'},
    q=99.2, cmap_num=2
)


### AID, the enzyme encoded by *AICDA*

is the single best marker of germinal-centre B cells, so your observation matches biology. "Only in tonsils", though, needs checking against how this reference was assembled: tissue here is confounded with study, donor age and 10x chemistry. I'll quantify it before critiquing:
- *AICDA* detection per tissue within each B subset, especially within GC B cells.
- How many GC B cells each tissue contributes at all.
- Whether the signal holds per donor and per chemistry (3′ vs 5′).
- Whether the "naive/memory" *AICDA*⁺ cells are single-UMI noise or GC-like cells under a non-GC label.

The data partly contradicts "only in tonsils". One more check, on the *AICDA*⁺ cells labelled memory or naive: do they carry a germinal-centre program, and are they likely doublets?

Your observation that *AICDA* is mostly in the dark zone (DZ) and cycling B cells is correct and fits the biology. "Only in tonsils" isn't: within germinal-centre (GC) B cells, the spleen and lymph nodes express *AICDA* at about the same rate. Tonsils dominate only because they contain almost all of the GC B cells in this reference.

#### What AICDA does

*AICDA* encodes **AID (activation-induced cytidine deaminase)**, an APOBEC-family enzyme. It converts cytosine to uracil in single-stranded DNA at actively transcribed immunoglobulin loci. That one reaction starts two processes:
- **Somatic hypermutation (SHM):** point mutations in the antibody variable regions. Proliferating DZ cells acquire the mutations, and light-zone (LZ) cells with higher affinity are selected. This drives affinity maturation.
- **Class-switch recombination (CSR):** double-strand breaks in switch regions change the antibody class from IgM/IgD to IgG, IgA or IgE. CSR can also happen outside germinal centres, in activated B cells at the T–B border.

Other points:
- **Regulation:** AID is induced by CD40L together with IL-4 or IL-21, and by TLR signals. It is tightly controlled after translation, through nuclear export and degradation.
- **Disease:** loss of AID causes hyper-IgM syndrome type 2 (no class switching, no hypermutation). Off-target AID activity causes oncogenic mutations and translocations such as *MYC–IGH* in Burkitt lymphoma and diffuse large B-cell lymphoma.

#### Critique

**1. "Only in tonsils" is mostly a cell-composition effect.** Half of tonsil B cells are GC B cells (DZ/LZ/prePB/Cycling), against 0.5–4% in the other tissues:

| Tissue | GC B cells | Share of B cells |
|---|---|---|
| Tonsil | 10,304 | 50% |
| tLN | 107 | 4.4% |
| Spleen | 122 | 2.9% |
| iLN | 69 | 2.0% |
| mLN | 35 | 0.5% |

That is why tonsils contribute 96% of all *AICDA*⁺ cells. Within GC B cells, per donor (≥ 20 GC cells each), detection is similar everywhere:
- tonsil donors: 21–47%;
- spleen: 42–46%;
- iLN: 27–30%;
- tLN: 19–27%.

So *AICDA* marks GC B cells in every tissue. Pediatric tonsils simply have many active germinal centres, consistent with constant antigen exposure in childhood.

**2. Tissue is confounded with donor, age and study.**
- All tonsils come from King 2020, from children.
- Spleen, iLN and tLN all come from one adult organ donor (A16, Park 2020).
- mLN comes from James 2020. It has only 35 GC B cells and no donor with ≥ 20, so it can't be evaluated.

Chemistry is not the explanation: within tonsil GC B cells, 3′ and 5′ data give 29.2% vs 29.3%.

**3. Detection is limited by sparse capture.** Between 52% and 79% of *AICDA*⁺ cells have a single UMI. So "41% of DZ cells" understates true expression, and a cell without *AICDA* reads may still express it. Ranking DZ above LZ (41% vs 9% in tonsil) is supported, and fits AID acting mainly during DZ proliferation. The exact percentages are partly set by how much RNA was captured.

**4. "A little in pre-GC" is not supported.** I found 0 of 404 `B_preGC` cells with any *AICDA* read.

**5. The *AICDA*⁺ naive and memory cells are probably not true naive or memory cells.**
- **"Naive":** 23 of 8,924 cells (0.26%). In tonsil, these 19 cells carry a full GC program, unlike the *AICDA*⁻ naive cells:

| Gene | *AICDA*⁺ "naive" | *AICDA*⁻ naive |
|---|---|---|
| *RGS13* | 68% | 2% |
| *BCL6* | 63% | 9% |
| *GCSAM* | 68% | 6% |
| *MKI67* | 58% | 0% |

  They look like mislabelled GC or cycling cells.
- **"Memory":** 96 of 13,476 cells (0.7%), 78 of them in tonsil. They show a partial GC program:

| Gene | *AICDA*⁺ memory | *AICDA*⁻ memory |
|---|---|---|
| *RGS13* | 24% | 9% |
| *LMO2* | 29% | 8% |
| *MKI67* | 19% | 1% |

  Plausible explanations: cells that recently left the GC (memory precursors), mislabelled LZ cells, or ambient RNA in tonsil suspensions, which are rich in GC transcripts.
- Their doublet scores are not elevated, but doublets were already filtered from this reference, so that check carries little weight.

**6. mRNA is not activity.** AID is regulated mostly after translation, so *AICDA* transcripts don't show that hypermutation or class switching is happening. This reference includes BCR data that could test it directly: `IGH_MU_FREQ` (IGH mutation frequency) and `ISOTYPE`. I haven't run that check.

**7. These are cell-level percentages from few donors.** For mLN and the A16 tissues, the GC B-cell counts are too small for firm conclusions.

I saved two tables in the output tray:
- `aicda_by_subset_tissue.csv` — % *AICDA*⁺ and cell counts for each B subset × tissue
- `aicda_gc_by_donor.csv` — % *AICDA*⁺ within GC B cells per donor, study and chemistry

In [ ]:
adata_ref.var.head(6).T.head(12)

In [ ]:
# var re-indexing (symbol column + Ensembl index) is done once in cell 34.
# Repeating it here would overwrite var['symbol'] with Ensembl IDs and raise KeyError: 'GeneID-2'.
assert "symbol" in adata_ref.var.columns and str(adata_ref.var_names[0]).startswith("ENSG"), "run cell 34 first"
adata_ref.var[["symbol"]].head(3)


In [ ]:
cols = ['symbol', 'GeneName-2', 'feature_types', 'feature_types-0', 'feature_types-1',
       'gene_ids-1', 'gene_ids-4861STDY7135913-0',
       'genome-1', 'n_cells', 'nonz_mean', 'mean_cov_effect_Subset_B_Cycling',
       'mean_cov_effect_Subset_B_GC_DZ', 
       'mean_sample_effectSample_BCP009_Total',
       'mean_sample_effectSample_Human_colon_16S7255677',
       'mean_sample_effectSample_Human_colon_16S7255678',
       'mean_sample_effectSample_Human_colon_16S8000484',
       'mean_sample_effectSample_Pan_T7935494',]

adata_ref.var[cols].head(3).T

<div class="alert alert-info">
<b>Note!</b>
    
Before we estimate the reference cell type signature we recommend to perform very permissive genes selection. We prefer this to standard highly-variable-gene selection because our procedure keeps markers of rare genes while removing most of the uninformative genes.
</div>

The default parameters **cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12** are a good starting point, however, you can increase the cut-off to exclude more genes. 

To preserve marker genes of rare cell types we recommend low **cell_count_cutoff=5**, however, **cell_percentage_cutoff2** and **nonz_mean_cutoff** can be increased to select between 8k-16k genes.

In this 2D histogram, orange rectangle highlights genes excluded based on the combination of number of cells expressing that gene (Y-axis) and average RNA count for cells where the gene was detected (X-axis).

In this case, the downloaded dataset was already filtered using this method, hence no density under the orange rectangle (to be changed in the future version of the tutorial).

### cell2location: "Markers of rare genes" is a slip in the original; it should read "markers of rare **cell types**".

#### The problem it solves

The reference regression model estimates an expression signature for every cell type. You want to:

- **remove** genes with almost no signal, which add noise and computation.
- **keep** genes that identify any cell type, including rare ones.

Standard highly-variable-gene (HVG) selection doesn't serve the second goal well:

- **It keeps a fixed top-N** (e.g. 2,000 genes) ranked by dispersion across *all* cells. A gene expressed only in a population making up 0.5% of cells adds little to the global variance, so it can fall below the cutoff.
- **It is tuned for a different job.** HVG selection picks genes for clustering and embedding. Cell2location wants as many informative genes as possible, because more genes give better deconvolution of spots.
- **It runs on normalised, log-transformed data.** Cell2location models raw counts.

#### What `filter_genes` does

```python
from cell2location.utils.filtering import filter_genes
selected = filter_genes(adata_ref, cell_count_cutoff=5,
                        cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
```

It works on raw counts. For each gene it computes two numbers:

- `n_cells`: how many cells have a count above 0.
- `nonz_mean`: the mean count *in those expressing cells only*.

A gene is kept if **either** rule applies:

1. **Broadly detected:** it is expressed in more than 3% of all cells (`cell_percentage_cutoff2`). This keeps genes expressed in common cell types.
2. **Rare but strongly expressed:** it is expressed in more than 5 cells (`cell_count_cutoff`) **and** its mean in the expressing cells is above 1.12 (`nonz_mean_cutoff`). This rule is how rare-cell-type markers survive. A gene found in only 200 cells, but at several UMIs per cell, looks like a real marker. A gene found in 200 cells at exactly 1 UMI each looks like ambient RNA or dropout noise.

Genes that meet neither rule are discarded. These are genes detected in 5 or fewer cells, or detected in under 3% of cells at only about 1 count each.

The value 1.12 is chosen just above 1. A gene passes rule 2 only if at least some of its expressing cells have 2 or more UMIs, which separates real expression from scattered single-molecule background.

#### Reading the diagnostic plot

The function draws a 2D histogram of genes: log10(`n_cells`) on the x-axis and log10(`nonz_mean`) on the y-axis.

- A vertical line marks the 3% threshold. Everything to its right is kept.
- To the left of that line, only genes above the horizontal 1.12 line are kept. These sit in the top-left region: few cells, but high counts.
- The large dense cloud at the bottom-left (low detection, about 1 count) is what gets removed.

#### In your lymphoid-organ reference

With 73,260 cells, 3% is about 2,198 cells, so rule 1 keeps genes detected in more than ~2,200 cells. Several genes from your table rely on rule 2 instead. For example, LILRA4 (pDCs), DERL3 (plasma cells) and AICDA (germinal-centre B cells) are expressed in small populations but strongly. HVG selection might drop them; this filter keeps them. I haven't run the filter on your data, so I can't confirm which of your genes actually pass.

The thresholds can be adjusted: raising `nonz_mean_cutoff` or `cell_percentage_cutoff2` gives stricter selection. I can run the filter on your reference and report which of the 40 genes in `selected_genes_lymphoid_organ.csv` survive.

In [ ]:
from cell2location.utils.filtering import filter_genes

selected = filter_genes(adata_ref, cell_count_cutoff=5, 
                        cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12,
                        figsize=(3,4))
# filter the object
adata_ref = adata_ref[:, selected].copy()

len(selected), len(adata_ref), selected[:10]

### Estimation of reference cell type signatures (NB regression)

The signatures are estimated from scRNA-seq data, accounting for batch effect, using a Negative binomial regression model.

<div class="alert alert-block alert-message">
<b>Preparing anndata.</b>
    
First, prepare anndata object for the regression model:
</div>

In [ ]:
adata_ref.obs.head(3)

In [ ]:
adata_ref.obsm

In [ ]:
adata_ref.var.head(3)

In [ ]:
adata_ref.var.shape

In [ ]:
adata_ref

### prepare anndata for the regression model

In [ ]:
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref,
                        # 10X reaction / sample / batch
                        batch_key='Sample',
                        # cell type, covariate used for constructing signatures
                        labels_key='Subset',
                        # multiplicative technical effects (platform, 3' vs 5', donor effect)
                        categorical_covariate_keys=['Method']
                       )

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training model.</b>
    
Now we train the model to estimate the reference cell type signatures.
    
Note that to achieve convergence on your data (=to get stabilization of the loss) you may need to increase `max_epochs=250` (See below).
    
Also note that here we are using `batch_size=2500` which is much larger than scvi-tools default and perform training on all cells in the data (`train_size=1`) - both parameters are defaults.
</div>

In [ ]:
adata_ref_filename = Path(f"{ref_run_name}/sc.h5ad")
adata_ref_filename.exists(), adata_ref_filename

In [ ]:
if not adata_ref_filename.exists():
    mod.train(max_epochs=250, accelerator='gpu')

    # In this section, we export the estimated cell abundance (summary of the posterior distribution).
    adata_ref = mod.export_posterior(
        adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500, 'accelerator': 'gpu'}
    )    

    # Save model
    mod.save(f"{ref_run_name}", overwrite=True, save_anndata=True)
    print(f"Model saved with anndata to {ref_run_name}")

    # Save anndata object with results
    adata_ref.write(adata_ref_filename)
    print(f"Model saved to {adata_ref_filename}")
else:
    # Load AnnData with posterior results (already registered via setup_anndata)
    # adata_ref = sc.read_h5ad(adata_ref_filename)

    # Reload the trained model bound to that AnnData
    mod = cell2location.models.RegressionModel.load(f"{ref_run_name}", adata_ref)
    adata_ref = mod.adata

    print(f"Model loaded from {ref_run_name}")

In [ ]:
adata_ref.var.shape

In [ ]:
import torch

saved = torch.load(f"{ref_run_name}/model.pt", map_location="cpu", weights_only=False)
model_genes = pd.Index(saved["var_names"])
adata_genes = adata_ref.var_names

print(len(model_genes), len(adata_genes))
print("missing from adata:", model_genes.difference(adata_genes)[:20].tolist())
print("extra in adata:", adata_genes.difference(model_genes)[:20].tolist())

<div class="alert alert-block alert-message">
<b>Determine if the model needs more training.</b>
</div>

Here, we plot ELBO loss history during training, removing first 20 epochs from the plot.
This plot should have a decreasing trend and level off by the end of training. If it is still decreasing, increase `max_epochs`.

In [ ]:
mod.plot_history(20)

You can compute the 5%, 50% and 95% quantiles of the posterior distribution directly rather than using 1000 samples from the distribution (or any other quantiles). This speeds up application on large datasets and requires less memory - however, posterior df_mean and standard deviation cannot be computed this way.

In [ ]:
list_q = ['q05_per_cluster_mu_fg', 'q50_per_cluster_mu_fg', 'q95_per_cluster_mu_fg', 'q0001_per_cluster_mu_fg']

ok = True if len([x for x in adata_ref.varm if x in list_q]) == len(list_q) else False

if not ok:
    adata_ref = mod_reg.export_posterior(
        adata_ref, use_quantiles=True,
        # choose quantiles
        add_to_varm=["q05","q50", "q95", "q0001"],
        sample_kwargs={'batch_size': 2500, 'accelerator': 'gpu'}
    )
else:
    print("export_posterior is ok: quantiles already exist in adata_ref.varm")

In [ ]:
# picks 'means', else 'q50', else 'q05' (export_posterior(use_quantiles=True) writes only quantiles)
inf_aver, summary = get_reference_signatures(adata_ref)
print(f"Using {summary} signatures")

inf_aver.head(6).T.head(8)


In [ ]:
factor_names = adata_ref.uns['mod']['factor_names']
inf_aver = adata_ref.varm['q50_per_cluster_mu_fg'][
    [f'q50_per_cluster_mu_fg_{i}' for i in factor_names]
].copy()

inf_aver.columns = factor_names

inf_aver.head(6).T.head(8)

<div class="alert alert-block alert-message">
<b>Examine QC plots.</b>
</div>

1. Reconstruction accuracy to assess if there are any issues with inference. This 2D histogram plot should have most observations along a noisy diagonal.

2. The estimated expression signatures are distinct from mean expression in each cluster because of batch effects. For scRNA-seq datasets which do not suffer from batch effect (this dataset does), cluster average expression can be used instead of estimating signatures with a model. When this plot is very different from a diagonal plot (e.g. very low values on Y-axis, density everywhere) it indicates problems with signature estimation.

In [ ]:
mod

In [ ]:
with plt.rc_context({"figure.figsize": (3,3)}):
    mod.plot_QC(summary_name="q50");    # median signatures

<div class="alert alert-block alert-message">
<b>Extracting reference cell types signatures as a pd.DataFrame.</b>
    
All parameters of the a Negative Binomial regression model are exported into reference anndata object, however for spatial mapping we just need the estimated expression of every gene in every cell type. Here we extract that from standard output:
</div>

In [ ]:
[x for x in adata_ref.var.columns if 'cluster' in x]

In [ ]:
# export estimated expression in each cluster
# (the original hard-coded 'means_per_cluster_mu_fg' -> KeyError after a quantile-only export)
inf_aver, summary = get_reference_signatures(adata_ref, prefer=("means", "q50"))
print(f"Using {summary} signatures")
inf_aver.iloc[0:5, 0:5]


### Cell2location: spatial mapping

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

In [ ]:
adata_vis.var_names[:10]

In [ ]:
inf_aver.head(3)

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

len(intersect), len(adata_vis.var_names), len(inf_aver.index)

#### prepare anndata for cell2location model

In [ ]:
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")

<div class="alert alert-info">
<b> Important </b>
    
To use cell2location spatial mapping model, you need to specify 2 user-provided hyperparameters (**N_cells_per_location** and **detection_alpha**) 

For detailed guidance on setting these hyperparameters and their impact see [the flow diagram and the note](https://github.com/BayraktarLab/cell2location/blob/master/docs/images/Note_on_selecting_hyperparameters.pdf).
</div>

**Choosing hyperparameter `N_cells_per_location`!**

It is useful to adapt the expected cell abundance `N_cells_per_location` to every tissue</b>. This value can be estimated from paired histology images and as described in the note above. Change the value presented in this tutorial (`N_cells_per_location=30`) to the value observed in your your tissue.

**Choosing hyperparameter `detection_alpha`!**

To improve accuracy & sensitivity on datasets with large technical variability in RNA detection sensitivity within the slide/batch - you need to relax regularisation of per-location normalisation (use `detection_alpha=20`). High technical variability in RNA detection sensitivity is present in your sample when you observe the spatial distribution of total RNA count per location that doesn't match expected cell numbers based on histological examination.

We initially opted for high regularisation (`detection_alpha=200`) as a default because the mouse brain & human lymph node datasets used in our paper have low technical effects and using high regularisation strenght improves consistencly between total estimated cell abundance per location and the nuclei count quantified from histology ([Fig S8F in cell2location paper](https://static-content.springer.com/esm/art%3A10.1038%2Fs41587-021-01139-4/MediaObjects/41587_2021_1139_MOESM1_ESM.pdf)). However, in many collaborations, we see that Visium experiments on human tissues suffer from technical effects. This motivates the new default value of `detection_alpha=20` and the recommendation of testing both settings on your data (`detection_alpha=20` and `detection_alpha=200`).

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    adata_vis, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)
mod.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Training cell2location:</b>
</div>

In [ ]:
adata_vis_filename_run = run_name / "sp.h5ad"
adata_vis_filename_run.exists(), adata_vis_filename_run


In [ ]:
if not adata_vis_filename_run.exists():
    mod.train(max_epochs=25000,
            # train using full data (batch_size=None)
            batch_size=None,
            # use all data points in training because
            # we need to estimate cell abundance at all locations
            train_size=1,
            accelerator='gpu',
            )

    # In this section, we export the estimated cell abundance (summary of the posterior distribution).
    adata_vis = mod.export_posterior(
        adata_vis, use_quantiles=True,
        add_to_obsm=["q05", "q50", "q95", "q0001"],
        sample_kwargs={'batch_size': mod.adata.n_obs, 'accelerator': 'gpu'} 
        # sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs, 'accelerator': 'gpu'}
    )

    # Compute expected expression per cell type
    expected_dict = mod.module.model.compute_expected_per_cell_type(
        mod.samples["post_sample_q05"], mod.adata_manager
    )

    # Add to anndata layers
    for i, n in enumerate(mod.factor_names_):
        adata_vis.layers[n] = expected_dict['mu'][i]

    # Save model
    mod.save(f"{run_name}", overwrite=True)
    print(f"Model saved to {run_name}")

    # Save anndata object with results
    adata_vis.write(adata_vis_filename_run)
    print(f"adata_vis saved to {adata_vis_filename_run}")
else:
    # Load AnnData with posterior abundances in .obsm (already registered via setup_anndata)
    adata_vis = sc.read_h5ad(adata_vis_filename_run)
    print(f"adata_vis read from {adata_vis_filename_run}")

    # Reload the trained spatial model bound to that AnnData
    mod = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)
    print(f"Model loaded from {run_name}")

    # mod.samples is not saved with the model: rebuild it (needed by plot_QC and compute_expected_per_cell_type)
    adata_vis = ensure_posterior_samples(mod, adata_vis, quantiles=["q05", "q50", "q95", "q0001"], accelerator="gpu")
    print("posterior samples:", sorted(mod.samples.keys()))

# Common to both branches: copy 5% quantile abundances into .obs for plotting
factor_names = adata_vis.uns['mod']['factor_names']
adata_vis.obs[factor_names] = adata_vis.obsm['q05_cell_abundance_w_sf']


In [ ]:
mod.adata.n_obs

### plot ELBO loss history during training, removing first 100 epochs from the plot

In [ ]:
mod.plot_history(1000)
plt.legend(labels=['full data training']);

<div class="alert alert-block alert-message">
<b>Exporting estimated posterior distributions of cell abundance and saving results:</b>
</div>

The model and output h5ad can be loaded later like this:
```python
adata_ref_filename = f"{run_name}/sp.h5ad"
adata_vis = sc.read_h5ad(adata_ref_filename)
mod = cell2location.models.Cell2location.load(f"{run_name}", adata_vis)
```

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [ ]:
with plt.rc_context({"figure.figsize": (3,3)}):
    mod.plot_QC(summary_name="q50");    # median signatures

When intergrating multiple spatial batches and when working with datasets that have substantial variation of detected RNA within slides (that cannot be explained by high cellular density in the histology), it is important to assess whether cell2location normalised those effects. You expect to see similar total cell abundance across batches but distinct RNA detection sensitivity (both estimated by cell2location). You expect total cell abundance to mirror high cellular density in the histology.

```python
fig = mod.plot_spatial_QC_across_batches()
```

## Visualising cell abundance in spatial coordinates <a class="anchor" id="Visualising-cell-abundance-in-spatial-coordinates"></a>

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present',
# to adata.obs with nice names for plottingp
adata_vis.obs[adata_vis.uns['mod']['factor_names']]= adata_vis.obsm['q05_cell_abundance_w_sf']

# select one slide
from cell2location.utils import select_slide
slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')


# plot in spatial coordinates
with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [4.5, 5]}):

    sc.pl.spatial(slide, cmap=cmap_byr,
                  # show first 8 cell types
                  color=['B_Cycling', 'B_GC_LZ', 'T_CD4+_TfH_GC', 'FDC',
                         'B_naive', 'T_CD4+_naive', 'B_plasma', 'Endo'],
                  ncols=4, size=1.3,
                  img_key='hires',
                  # limit color scale at 99.2% quantile of cell abundance
                  vmin=0, vmax='p99.2'
                 )

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

# Now we use cell2location plotter that allows showing multiple cell types in one panel
C2L_COLORS = ["#F0E442", "#D55E00", "#56B4E9", "#009E73", "#5A14A5", "#C8C8C8", "#323232"]  # plot_spatial order

clust_labels = ['T_CD4+_naive', 'B_naive', 'FDC']

c2l.plot_slide_loc_cell_types(slide, clust_labels, C2L_COLORS, root_figure);

### B-INF -  IFIT1 and IFIT3 signatures

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

ISG_GENES = ["IFIT3","IFIT1","IFIT2","IFI44L","ISG15","MX1","MX2","IFI6","RSAD2","OAS1","STAT1","XAF1"]

hot_res = c2l.run_isg_hotspot(adata_vis, ISG_GENES, root_results,root_figure);

In [ ]:
hot_res

### Downstream analysis <a class="anchor" id="Downstream-analysis"></a>

#### Identifying discrete tissue regions by Leiden clustering<a class="anchor" 

id="Identifying-discrete-tissue-regions-by-Leiden-clustering"></a>

We identify tissue regions that differ in their cell composition by clustering locations using cell abundance estimated by cell2location.

We find tissue regions by clustering Visium spots using estimated cell abundance each cell type. We constuct a K-nearest neigbour (KNN) graph representing similarity of locations in estimated cell abundance and then apply Leiden clustering. The number of KNN neighbours should be adapted to size of dataset and the size of anatomically defined regions (e.i. hippocampus regions are rather small compared to size of the brain so could be masked by large `n_neighbors`). This can be done for a range KNN neighbours and Leiden clustering resolutions until a clustering matching the anatomical structure of the tissue is obtained.

The clustering is done jointly across all Visium sections / batches, hence the region identities are directly comparable. When there are strong technical effects between multiple batches (not the case here) `sc.external.pp.bbknn` can be in principle used to account for those effects during the KNN construction.

The resulting clusters are saved in `adata_vis.obs['region_cluster']`.

In [ ]:
# compute KNN using the cell2location output stored in adata.obsm
sc.pp.neighbors(adata_vis, use_rep='q05_cell_abundance_w_sf', n_neighbors = 15)

# Cluster spots into regions using scanpy
sc.tl.leiden(adata_vis, resolution=1.1)

# add region as categorical variable
adata_vis.obs["region_cluster"]= adata_vis.obs["leiden"].astype("category")

adata_vis

We can use the location composition similarity graph to build a joint integrated UMAP representation of all section/Visium batches.

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

c2l.umap_spatial_3plot(adata_vis, root_figure);

### Grouped the panel genes into five cell-type markers:

1. pan-B (CD19/MS4A1/CD79A/B/PAX5/CD22)
2. follicular mantle B (IGHD/IGHM)
3. pan-lymphocyte (CD52/CD53)
4. TIM3+ T, GC TfH, Treg, GC DZ B (CD27)
5. monocytes/macrophages (CD14) 

CDS2 doesn't mark any clear cell type.

In [ ]:
ct = adata_ref.obs["Subset"].astype(str).values
np.unique(ct)

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

# row order for the dot plot: cell types grouped by lineage
lineage = {
    "GC B":        ["B_GC_DZ","B_GC_LZ","B_GC_prePB","B_Cycling","B_preGC"],
    "naive/mem B": ["B_naive","B_activated","B_IFN","B_mem"],
    "plasma":      ["B_plasma"],
    "T":           ["T_CD4+_naive","T_CD4+","T_CD4+_TfH","T_CD4+_TfH_GC","T_TfR","T_Treg","T_TIM3+",
                    "T_CD8+_naive","T_CD8+_cytotoxic","T_CD8+_CD161+"],
    "NK/ILC":      ["NK","NKT","ILC"],
    "myeloid":     ["Monocytes","Macrophages_M1","Macrophages_M2","DC_cDC1","DC_cDC2","DC_pDC","DC_CCR7+","Mast"],
    "stromal":     ["FDC","Endo","VSMC"],
}

focus=["B_GC_LZ","B_GC_DZ","B_Cycling","B_preGC","B_naive","B_mem","B_plasma","T_CD4+_TfH_GC","T_TIM3+","FDC","Macrophages_M1","Monocytes"]


# Figure 1: UMAP by cell type + dot plot
dfg = c2l.umap_cell_type_plot(adata_vis, adata_ref, genes_plot, lineage, focus, root_figure)

In [ ]:
print(dfg.shape)
dfg.head(4)

### Figure 2: spatial maps

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

c2l.spatial_plot_bcell_types(adata_vis, focus, lineage, root_figure)

### Figure 3: gene (Visium) vs cell-type abundance correlation

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
%matplotlib inline

df_rho = c2l.spatial_plot_cell_abundance(adata_vis, adata_ref, genes_plot, focus, lineage, root_results, root_figure)

In [ ]:
df_rho.head(6)

### Identifying cellular compartments / tissue zones using matrix factorisation (NMF) <a name="Identifying-cellular-compartments-/-tissue-zones-using-matrix-factorisation-(NMF)"></a>

Here, we use the cell2location mapping results to identify the spatial co-occurrence of cell types in order to better understand the tissue organisation and predict cellular interactions. We performed non-negative matrix factorization (NMF) of the cell type abundance estimates from cell2location ([paper section 4, Fig 4D](https://www.nature.com/articles/s41587-021-01139-4)). Similar to the established benefits of applying NMF to conventional scRNA-seq, the additive NMF decomposition yielded a grouping of spatial cell type abundance profiles into components that capture co-localised cell types ([Supplemenary Methods section 4.2, p. 60](https://www.nature.com/articles/s41587-021-01139-4#Sec50)). This NMF-based decomposition naturally accounts for the fact that multiple cell types and microenvironments can co-exist at the same Visium locations (see [paper Fig S20, p. 34](https://www.nature.com/articles/s41587-021-01139-4#Sec50)), while sharing information across tissue areas (e.g. individual germinal centres).


<div class="alert alert-block alert-primary">
<b>Tip</b>
    
In practice, it is better to train NMF for a range of factors $R={5, .., 30}$ and select $R$ as a balance between capturing fine-grained and splitting known well-established tissue zones.

If you want to find a few most disctinct cellular compartments, use a small number of factors.   
If you want to find very strong co-location signal and assume that most cell types don't co-locate, use a lot of factors (> 30 - used here).
</div>

Below we show how to perform this analysis. To aid this analysis, we wrapped the analysis shown the notebook on advanced downstream analysis into a pipeline that automates training of the NMF model with varying number of factors:

In [ ]:
from cell2location import run_colocation
import pickle

fname = "res_dict_coloc.pkl"
filename = root_results / fname

if filename.exists():
    with open(filename, "rb") as fh:
      res_dict = pickle.load(fh)
    print(f"Results loaded from {filename}")
else:
    res_dict, adata_vis = run_colocation(
        adata_vis,
        model_name='CoLocatedGroupsSklearnNMF',
        train_args={
          'n_fact': np.arange(11, 13), # IMPORTANT: use a wider range of the number of factors (5-30)
          'sample_name_col': 'sample', # columns in adata_vis.obs that identifies sample
          'n_restarts': 3 # number of training restarts
        },
        # the hyperparameters of NMF can be also adjusted:
        model_kwargs={'alpha': 0.01, 'init': 'random', "nmf_kwd_args": {"tol": 0.000001}},
        export_args={'path': f'{run_name}/CoLocatedComb/'}
    )

    with open(filename, "wb") as fh:
      pickle.dump(res_dict, fh)
    print(f"Results saved to {filename}")

For every factor number, the model produces the following list of folder outputs:

`cell_type_fractions_heatmap/`: a dot plot of the estimated NMF weights of cell types (rows) across NMF components (columns)

`cell_type_fractions_mean/`: the data used for dot plot

`factor_markers/`: tables listing top 10 cell types most speficic to each NMF factor

`models/`: saved NMF models

`predictive_accuracy/`: 2D histogram plot showing how well NMF explains cell2location output

`spatial/`: NMF weights across locatinos in spatial coordinates

`location_factors_mean/`: the data used for the plot in spatial coordiantes

`stability_plots/`: stability of NMF weights between training restarts

Key output that you want to examine are the files in `cell_type_fractions_heatmap/` which show a dot plot of the estimated NMF weights of cell types (rows) across NMF components (columns) which correspond to cellular compartments. Shown are relative weights, normalized across components for every cell type.

<div class="alert alert-block alert-primary">
<b>Tip</b>
    
The NMF model output such as factor loadings are stored in `adata.uns[f"mod_coloc_n_fact{n_fact}"]` in a similar output format as main cell2location results in `adata.uns['mod']`.
</div>

In [ ]:
# Here we plot the NMF weights (Same as saved to `cell_type_fractions_heatmap`)
res_dict['n_fact12']['mod'].plot_cell_type_loadings()

### Estimate cell-type specific expression of every gene in the spatial data (needed for NCEM) <a name="Estimate-cell-type-specific-expression-of-every-gene-in-the-spatial-data-(needed-for-NCEM)"></a>

The cell-type specific expression of every gene at every spatial location in the spatial data enables learning cell communication with NCEM model using Visium data (https://github.com/theislab/ncem).

To derive this, we adapt the approach of estimating conditional expected expression proposed by [RCTD (Cable et al)](https://pubmed.ncbi.nlm.nih.gov/33603203/) method.

With cell2location, we can look at the posterior distribution rather than just point estimates of cell type specific expression (see `mod.samples.keys()` and next section on using full distribution).

Note that this analysis requires substantial amount of RAM memory and thefore doesn't work on free Google Colab (12 GB limit).

In [ ]:
mod

In [ ]:
print(list(mod.samples.keys()))

In [ ]:
adata_vis.layers

In [ ]:
'''
# Compute expected expression per cell type
expected_dict = mod.module.model.compute_expected_per_cell_type(
    mod.samples["post_sample_q05"], mod.adata_manager
)

# Add to anndata layers
for i, n in enumerate(mod.factor_names_):
    adata_vis.layers[n]= expected_dict['mu'][i]
'''

# Save anndata object with results (adata_vis_filename_run, not adata_ref_filename, which is the reference sc.h5ad path)
adata_vis.write(adata_vis_filename_run)
adata_vis_filename_run

<div class="alert alert-block alert-primary">
<b>Plotting cell-type specific expression of genes in spatial coordinates.</b>
    
Below we plot the cell-type specific expression of genes (rows, second to last columns) compared to total expression of those genes (first column).

Here we highlight *CD3D*, pan T-cell marker expressed by 2 subtypes of T cells in distinct locations but not expressed by co-located B cells, that instead express *CR2* gene.
</div>

In [ ]:
import libs.cell2location_lib as c2l
importlib.reload(c2l)

In [ ]:
# list cell types and genes for plotting
ctypes = ['T_CD4+_TfH_GC', 'T_CD4+_naive', 'B_GC_LZ']
genes = ['CD3D', 'CR2']

with mpl.rc_context({'axes.facecolor':  'black'}):
    # select one slide
    slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

    # from tutorial_utils import plot_genes_per_cell_type
    c2l.plot_genes_per_cell_type(slide, genes, ctypes)

In [ ]:
slide = select_slide(adata_vis, 'V1_Human_Lymph_Node')

In [ ]:
print(slide.var.shape)
slide.var.head(3)

In [ ]:
slide.layers

Note that `plot_genes_per_cell_type` function often need customization so it is not included into cell2location package - you need to copy it from https://github.com/BayraktarLab/cell2location/blob/master/docs/notebooks/tutorial_utils.py to use on your system.

In [ ]:
cell2location.__version__